## Tính embedding R2Vul trên Colab T4: UniXcoder, vector cấp nút
Cần thư mục `Drive của tôi/hocmay_r2vul` chứa `colab_pack_r2vul.tar.gz`. Kết quả lưu theo từng phần trong `hocmay_r2vul/embeddings`; phiên bị ngắt thì chạy lại, phần đã xong được bỏ qua.


In [ ]:
# Nguyễn Tâm Bình - 2324801030003
TASKS = [("r2vul", "node", "unixcoder")]
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/hocmay_r2vul")
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))


### Mount Drive và chuẩn bị dữ liệu
Gói dữ liệu chỉ giải nén một lần trong mỗi phiên. Kết quả lưu ở `DRIVE/embeddings`.

In [ ]:
# Nguyễn Tâm Bình - 2324801030003
from google.colab import drive
drive.mount("/content/drive")
DRIVE.mkdir(parents=True, exist_ok=True)
import tarfile
WORK = Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)
marker = WORK / ".colab_pack_ready"
if not marker.exists():
    with tarfile.open(DRIVE / "colab_pack_r2vul.tar.gz", "r:gz") as archive:
        archive.extractall(WORK, filter="data")
    marker.touch()
else:
    print("Đã giải nén, bỏ qua.")
(DRIVE / "embeddings").mkdir(parents=True, exist_ok=True)
link = WORK / "data/embeddings"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    assert link.resolve() == (DRIVE / "embeddings").resolve(), "Symlink embeddings đang trỏ sai thư mục."
elif link.exists():
    raise RuntimeError("data/embeddings đã tồn tại và không phải symlink; kiểm tra trước khi chạy.")
else:
    link.symlink_to(DRIVE / "embeddings", target_is_directory=True)
import importlib.util
if importlib.util.find_spec("yaml") is None:
    !pip install -q pyyaml
print("Checkpoint:", link.resolve())


### Tính embedding
Qwen2.5-Coder-1.5B cho vector cấp hàm, UniXcoder cho vector cấp nút. Phần đã hoàn thành được bỏ qua.

In [ ]:
# Nguyễn Tâm Bình - 2324801030003
import shutil
def clean(folder):
    for done in folder.glob('*.done'):
        part = folder / 'parts' / done.stem
        if part.is_dir():
            shutil.rmtree(part)
for dataset, mode, model in TASKS:
    batch = "16" if model == "qwen15" else "512"
    code = get_ipython().system(f"cd /content/work && EMBED_DEVICE=cuda EMBED_BATCH={batch} PYTHONPATH=src python -m vuldet.llm.embed --dataset {dataset} --mode {mode} --model {model} 2>&1")
    if get_ipython().user_ns.get("_exit_code"):
        raise RuntimeError(f"Lỗi khi tính {dataset} {mode} {model}, xem log ngay phía trên.")
    clean(WORK / 'data/embeddings' / dataset / (model if model != "unixcoder" else ""))


### Kiểm tra kết quả
Em đối chiếu số dòng, kiểu float16 và giá trị NaN/Inf trước khi dùng kết quả.

In [ ]:
# Nguyễn Tâm Bình - 2324801030003
import numpy as np
def line_count(path):
    with path.open(encoding="utf-8") as stream:
        return sum(1 for _ in stream)
def check_array(path, count):
    values = np.load(path, mmap_mode="r", allow_pickle=False)
    assert values.ndim == 2 and values.shape[0] == count, str(path)
    assert values.dtype == np.float16, str(path)
    for start in range(0, len(values), 2000):
        assert np.isfinite(values[start:start + 2000]).all(), str(path)
    print(path.relative_to(WORK), values.shape)
for dataset, mode, model in TASKS:
    if mode == "func":
        folder = WORK / "data/embeddings" / dataset / model
        for split in ("train", "valid", "test"):
            count = line_count(WORK / "data/interim" / dataset / f"{split}.jsonl")
            assert line_count(folder / f"func_{split}_ids.txt") == count, f"{dataset}/{split}: lệch số ID"
            check_array(folder / f"func_{split}.npy", count)
            assert (folder / f"func_{split}.done").is_file()
    else:
        folder = WORK / "data/embeddings" / dataset
        check_array(folder / "node_codes.npy", line_count(WORK / "data/graphs" / dataset / "node_codes.txt"))
        assert (folder / "node_codes.done").is_file()
print("Kiểm tra đạt.")


### Xong
Kết quả nằm trong thư mục `embeddings` trên Drive, em tải về máy chủ để huấn luyện mô hình.


In [ ]:
# Nguyễn Tâm Bình - 2324801030003
print("Xong:", TASKS, "| Thư mục kết quả:", DRIVE / "embeddings")
